# EXP-005D2 — Scoring-Time Artifact Capture Proof

Diagnostic-only notebook after repeated placeholder artifacts in EXP-005B/005C/005D.

Goal: prove whether files written inside `KAGGLE_IS_COMPETITION_RERUN` and inside `MyAgent` during `main.py --agent myagent` are exposed in Kaggle output after competition scoring.

This notebook intentionally does **not** change planner logic and should not be treated as a leaderboard-improvement experiment.

Current best reference remains:

```text
EXP-005A / V9 public score: 0.17
```

Recent regressions / diagnostics:

```text
EXP-005B: 0.09
EXP-005C / V12: 0.10
EXP-005D / V13: 0.10
EXP-005D / V14: 0.10
```

Expected proof artifacts:

```text
/kaggle/working/EXP005D2_RERUN_BRANCH_PROOF.txt
/kaggle/working/EXP005D2_AGENT_INIT_PROOF.txt
/kaggle/working/EXP005D2_AFTER_MAIN_PROOF.txt
/kaggle/working/EXP005D2_NON_RERUN_PROOF.txt
/kaggle/working/exp005d2_rerun_branch_proof.json
/kaggle/working/exp005d2_agent_events.jsonl
/kaggle/working/exp005d2_run_summary.json
/kaggle/working/exp005d2_artifact_inventory.json
```


In [ ]:
!pip install -q --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
import json
import os
import random
import time
import traceback
from pathlib import Path

import numpy as np

from agents.agent import Agent
from arcengine import GameAction, GameState

VARIANT = "EXP-005D2"
ROOT = Path("/kaggle/working")
EVENT_PATH = ROOT / "exp005d2_agent_events.jsonl"
SUMMARY_PATH = ROOT / "exp005d2_run_summary.json"
AGENT_INIT_PROOF = ROOT / "EXP005D2_AGENT_INIT_PROOF.txt"


def now_record(event, **kwargs):
    rec = {
        "variant": VARIANT,
        "event": event,
        "time": time.time(),
        "pid": os.getpid(),
        "competition_rerun_env": bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN")),
        **kwargs,
    }
    try:
        EVENT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with EVENT_PATH.open("a") as f:
            f.write(json.dumps(rec, default=str) + "\n")
    except Exception:
        pass
    return rec


def save_summary(**kwargs):
    payload = {
        "variant": VARIANT,
        "time": time.time(),
        "pid": os.getpid(),
        "competition_rerun_env": bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN")),
        **kwargs,
    }
    try:
        SUMMARY_PATH.write_text(json.dumps(payload, indent=2, default=str))
    except Exception:
        pass


def action_from_id(action_id):
    return GameAction.from_id(int(action_id))


class MyAgent(Agent):
    MAX_ACTIONS = float("inf")
    _MAX_FRAMES = 10

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.start = time.time()
        self.rng = random.Random(5052)
        self.action_count = 0
        self.levels_seen = set()
        self.policy_counts = {
            "reset": 0,
            "click": 0,
            "move": 0,
            "fallback": 0,
            "error_fallback": 0,
        }

        AGENT_INIT_PROOF.write_text(
            "EXP-005D2: written inside MyAgent.__init__ during main.py --agent myagent.\n"
            f"KAGGLE_IS_COMPETITION_RERUN={os.getenv('KAGGLE_IS_COMPETITION_RERUN')}\n"
            f"game_id={getattr(self, 'game_id', None)}\n"
            f"pid={os.getpid()}\n"
            f"time={time.time()}\n"
        )

        now_record(
            "agent_init",
            game_id=getattr(self, "game_id", None),
            arc_env_present=hasattr(self, "arc_env"),
        )
        self.write_summary("agent_init")

    def write_summary(self, event):
        save_summary(
            event=event,
            game_id=getattr(self, "game_id", None),
            action_count=self.action_count,
            levels_seen=sorted(list(self.levels_seen)),
            policy_counts=dict(self.policy_counts),
            elapsed=round(time.time() - self.start, 3),
            proof_files_expected=[
                str(AGENT_INIT_PROOF),
                str(EVENT_PATH),
                str(SUMMARY_PATH),
            ],
        )

    def append_frame(self, frame):
        self.frames.append(frame)
        self.frames = self.frames[-self._MAX_FRAMES :]
        if getattr(frame, "guid", None):
            self.guid = frame.guid
        if hasattr(self, "recorder") and not self.is_playback:
            try:
                self.recorder.record(json.loads(frame.model_dump_json()))
            except Exception:
                pass

    def _level(self, latest_frame):
        return int(getattr(latest_frame, "levels_completed", 0) or 0)

    def _available(self, latest_frame):
        out = []
        for action in getattr(latest_frame, "available_actions", None) or []:
            try:
                value = int(action.value)
            except Exception:
                try:
                    value = int(action)
                except Exception:
                    continue
            if value not in out:
                out.append(value)
        return out

    def _frame_array(self, latest_frame):
        frame = getattr(latest_frame, "frame", None)
        if frame is None:
            return np.zeros((64, 64), dtype=np.int64)
        arr = np.asarray(frame)
        if arr.ndim == 3:
            return arr[-1].astype(np.int64)
        if arr.ndim == 2:
            return arr.astype(np.int64)
        return np.zeros((64, 64), dtype=np.int64)

    def _visible_click(self, latest_frame):
        arr = self._frame_array(latest_frame)
        bg = int(np.bincount(arr.flatten(), minlength=16).argmax())
        ys, xs = np.where(arr != bg)
        if len(xs) == 0:
            return None
        idx = self.rng.randrange(len(xs))
        return {
            "x": int(xs[idx]),
            "y": int(ys[idx]),
            "game_id": "exp005d2_artifact_capture",
        }

    def choose_action(self, frames, latest_frame):
        try:
            self.action_count += 1
            level = self._level(latest_frame)
            self.levels_seen.add(level)
            state = getattr(latest_frame, "state", None)
            available = self._available(latest_frame)

            if self.action_count <= 10 or self.action_count % 25 == 0:
                now_record(
                    "choose_action",
                    action_count=self.action_count,
                    level=level,
                    state=str(state),
                    available_actions=available,
                )

            if state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self.policy_counts["reset"] += 1
                action = GameAction.RESET
                action.reasoning = "exp005d2:reset"
                self.write_summary("reset")
                return action

            if 6 in available:
                click = self._visible_click(latest_frame)
                if click is not None:
                    action = GameAction.ACTION6
                    action.set_data(click)
                    action.reasoning = "exp005d2:visible_click_artifact_capture"
                    self.policy_counts["click"] += 1
                    self.write_summary("click")
                    return action

            moves = [a for a in available if 1 <= a <= 5]
            if moves:
                selected = self.rng.choice(moves)
                action = action_from_id(selected)
                action.reasoning = "exp005d2:move_artifact_capture"
                self.policy_counts["move"] += 1
                self.write_summary("move")
                return action

            self.policy_counts["fallback"] += 1
            action = GameAction.RESET
            action.reasoning = "exp005d2:fallback_reset"
            self.write_summary("fallback_reset")
            return action

        except Exception as exc:
            traceback.print_exc()
            self.policy_counts["error_fallback"] += 1
            now_record("error_fallback", error=repr(exc))
            self.write_summary("error_fallback")
            action = GameAction.RESET
            action.reasoning = f"exp005d2:error_fallback:{exc}"
            return action

    def is_done(self, frames, latest_frame):
        try:
            done = latest_frame.state is GameState.WIN or time.time() - self.start >= 8 * 3600 - 300
            if done:
                now_record("is_done", action_count=self.action_count, state=str(getattr(latest_frame, "state", None)))
                self.write_summary("is_done")
            return done
        except Exception:
            self.write_summary("is_done_exception")
            return True


In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

ROOT = Path("/kaggle/working")
VARIANT = "EXP-005D2"

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    (ROOT / "EXP005D2_RERUN_BRANCH_PROOF.txt").write_text(
        "EXP-005D2: written inside the KAGGLE_IS_COMPETITION_RERUN branch before main.py.\n"
        f"time={time.time()}\n"
        f"pid={os.getpid()}\n"
    )
    (ROOT / "exp005d2_rerun_branch_proof.json").write_text(json.dumps({
        "variant": VARIANT,
        "mode": "competition_rerun_branch",
        "event": "before_main_py",
        "time": time.time(),
        "pid": os.getpid(),
        "KAGGLE_IS_COMPETITION_RERUN": os.getenv("KAGGLE_IS_COMPETITION_RERUN"),
    }, indent=2))

    BASE = ROOT / "ARC-AGI-3-Agents"
    SRC = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    AGENT_SRC = ROOT / "my_agent.py"
    AGENT_DST = BASE / "agents/templates/my_agent.py"

    subprocess.run(
        [
            "curl",
            "--fail",
            "--retry",
            "60",
            "--retry-all-errors",
            "--retry-delay",
            "5",
            "--retry-max-time",
            "300",
            "http://gateway:8001/api/games",
        ],
        check=False,
    )

    if BASE.exists():
        shutil.rmtree(BASE)
    shutil.copytree(SRC, BASE)
    shutil.copy(AGENT_SRC, AGENT_DST)

    (BASE / "agents/__init__.py").write_text(
        'from typing import Type\n'
        'from dotenv import load_dotenv\n'
        'from .agent import Agent, Playback\n'
        'from .swarm import Swarm\n'
        'from .templates.random_agent import Random\n'
        'from .templates.my_agent import MyAgent\n'
        'load_dotenv()\n'
        'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}\n'
    )
    (BASE / ".env").write_text(
        "\n".join(
            [
                "SCHEME=http",
                "HOST=gateway",
                "PORT=8001",
                "ARC_API_KEY=test-key-123",
                "ARC_BASE_URL=http://gateway:8001/",
                "OPERATION_MODE=online",
                "RECORDINGS_DIR=/kaggle/working/server_recording",
            ]
        )
        + "\n"
    )

    subprocess.run(
        ["python", "main.py", "--agent", "myagent"],
        cwd=str(BASE),
        env={**os.environ, "MPLBACKEND": "agg", "PYTHONUNBUFFERED": "1"},
        check=True,
    )

    (ROOT / "EXP005D2_AFTER_MAIN_PROOF.txt").write_text(
        "EXP-005D2: written after main.py --agent myagent completed inside rerun branch.\n"
        f"time={time.time()}\n"
        f"pid={os.getpid()}\n"
    )

else:
    (ROOT / "EXP005D2_NON_RERUN_PROOF.txt").write_text(
        "EXP-005D2: written in visible notebook/non-rerun mode only.\n"
        "If this is the only proof file exposed after scoring, Kaggle output is not exposing rerun artifacts.\n"
        f"time={time.time()}\n"
        f"pid={os.getpid()}\n"
    )
    print("Local/non-rerun mode: wrote EXP005D2_NON_RERUN_PROOF.txt")


In [ ]:
import os
from pathlib import Path

if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    import pandas as pd

    pd.DataFrame(
        [
            {
                "row_id": "exp005d2_debug_0",
                "game_id": "artifact_capture_proof",
                "end_of_game": True,
                "score": 0,
            }
        ]
    ).to_parquet("/kaggle/working/submission.parquet", index=False)
    print("Wrote /kaggle/working/submission.parquet dummy fallback.")


In [ ]:
import json
import os
from pathlib import Path

ROOT = Path("/kaggle/working")
patterns = [
    "EXP005D2_*",
    "exp005d2_*",
    "submission.parquet",
]
inventory = []
for pattern in patterns:
    for path in sorted(ROOT.glob(pattern)):
        if path.is_file():
            inventory.append(
                {
                    "path": str(path),
                    "name": path.name,
                    "size": path.stat().st_size,
                    "mode": "non_rerun_visible" if not os.getenv("KAGGLE_IS_COMPETITION_RERUN") else "competition_rerun",
                }
            )

(ROOT / "exp005d2_artifact_inventory.json").write_text(json.dumps(inventory, indent=2))

print("EXP-005D2 artifact inventory:")
for item in inventory:
    print(f" - {item['path']} size={item['size']} mode={item['mode']}")


## Tracker row draft

```text
| EXP-005D2 | 2026-05-24 | `notebooks/01_exploration/exp005d2_scoring_artifact_capture_proof.ipynb` | Scoring-time artifact capture proof | Diagnostic-only notebook; writes distinct proof files in visible non-rerun mode, rerun branch before main.py, MyAgent.__init__, and after main.py | Pending | Pending | Created / not a score-improvement experiment | Goal: determine whether scoring-time artifacts are exposed after Kaggle code-submission scoring. |
```

## Validation rule

After Kaggle scoring, inspect downloadable output artifacts.

If files like these appear:

```text
EXP005D2_RERUN_BRANCH_PROOF.txt
EXP005D2_AGENT_INIT_PROOF.txt
EXP005D2_AFTER_MAIN_PROOF.txt
exp005d2_agent_events.jsonl
exp005d2_run_summary.json
```

then scoring-time artifacts are exposed and future diagnostics can rely on `/kaggle/working` JSON files.

If only this file appears:

```text
EXP005D2_NON_RERUN_PROOF.txt
```

then Kaggle output is exposing visible notebook artifacts only, and future diagnostics must use indirect score-delta or recording-based methods.
